In [1]:
import math
import torch
import torch.nn
import torch.nn.functional as F
from dataclasses import dataclass
from typing import Optional, Tuple

from jupyterlab.semver import loose_key_function
from numpy.ma.core import ndim
from torch import nn
from torchgen.gen_functionalization_type import return_from_mutable_noop_redispatch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cpu


#### LLama model config

In [2]:
@ dataclass
class LLaMaConfig:
    hidden_dim: int = 4096
    n_layers: int = 32
    n_heads: int = 32
    n_kv_heads:int = 32
    vocab_size:int = 32000
    max_seq_length: int = 2048
    norm_eps: float = 1e-6
    head_dim: int = hidden_dim // n_heads

    @property
    def ffn_dim(self) -> int:
        hidden = int(self.hidden_dim * 8 / 3)
        return ((hidden + 255) // 256) * 256


config = LLaMaConfig(
    hidden_dim=512,
    n_layers=8,
    n_heads=8,
    n_kv_heads=4,
    vocab_size=32000,
    max_seq_length=512
)

#### RMSNorm

In [3]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps:float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def norm(self,x: torch.Tensor) -> torch.Tensor:
        return x * torch.rsqrt(x.pow(2).mean(-1,keepdim=True)+ self.eps)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        norm = self.norm(x)
        return norm * self.weight

#### RoPE

In [4]:
def computer_freqs_cis(dim:int, seq_len:int ,theta:float =10000.0):
    # theta_k = 10000^{-2k/d}, k = 0, 1, ..., d/2-1
    freqs = 1.0 / (theta **(torch.arange(0, dim, 2).float() / dim))
    t = torch.arange(seq_len,dtype = torch.float32)
    # (seq_len,) × (dim/2,) → (seq_len, dim/2)
    freqs = torch.outer(t, freqs)
    # freqs_cis: e^{i * m * theta_k}
    freqs_cis = torch.polar(torch.ones_like(freqs), freqs)
    return freqs_cis

def reshape_for_broadcast(freqs_cis:torch.Tensor, x:torch.Tensor) -> torch.Tensor:
    ### reshape the shape of freqs_cis to (batch, seq_len, n_heads, head_dim/2)
    ndim = x.ndim
    assert ndim >=2
    shape = [1] * ndim
    shape[1] = x.shape[1]
    shape[-1] = x.shape[-1]
    return freqs_cis.view(*shape)

def apply_rotary_emb(q:torch.Tensor, k:torch.Tensor, freqs_cis:torch.Tensor) -> torch.Tensor:
    """
    对 Q 和 K 应用旋转位置编码。

    Args:
        q: (batch, seq_len, n_heads, head_dim)
        k: (batch, seq_len, n_kv_heads, head_dim)
        freqs_cis: (seq_len, head_dim//2) complex

    Returns:
        xq_out, xk_out: 旋转后的 Q, K
    """
    q_HalfDim =  q.float().reshape(*q.shape[:-1], -1, 2)
    k_HalfDim =  k.float().reshape(*k.shape[:-1], -1, 2)
    #k_HalfDim: (batch, seq_len, n_heads, head_dim/2, 2)
    #q_HalfDim: (batch, seq_len, n_kv_heads, head_dim/2, 2)

    q_complex = torch.view_as_complex(q_HalfDim)  # (B, T, n_heads, D/2)
    k_complex = torch.view_as_complex(k_HalfDim)  # (B, T, n_kv_heads, D/2)

    #reshape the shape of freqs_cis
    freqs_cis_q = reshape_for_broadcast(freqs_cis, q_complex)
    freqs_cis_k = reshape_for_broadcast(freqs_cis, k_complex)

    #rotate
    q_rotated = torch.view_as_real(q_complex * freqs_cis_q)
    # (B, T, n_heads, D/2, 2)
    k_rotated = torch.view_as_real(k_complex * freqs_cis_k)
    # (B, T, n_kv_heads, D/2, 2)

    q_out = q_rotated.reshape(*q.shape)
    # (B, T, n_heads, D/2, 2)
    k_out = k_rotated.reshape(*k.shape)
    # (B, T, n_kv_heads, D)
    return q_out, k_out


#### SwiGlu FFN


In [5]:
class SwiGlu_FFN(nn.Module):
    def __init__(self, dim:int, hidden_dim:int):
        super().__init__()
        self.w_gate = nn.Linear(dim, hidden_dim, bias=False)
        self.w_up = nn.Linear(dim, hidden_dim, bias=False)
        self.w_down = nn.Linear(hidden_dim, dim, bias=False)
        self.silu = nn.SiLU()
    def forward(self, x:torch.Tensor) -> torch.Tensor:
        gate = self.w_gate(x)
        up = self.w_up(x)
        gate = self.silu(gate)
        output = self.w_down(gate * up)
        return output

#### Grouped query attention

In [6]:
def repeat_kv(x: torch.Tensor, n_rep: int) -> torch.Tensor:
    """
    将 KV 头重复 n_rep 次以匹配 Q 头数。

    Args:
        x: (batch, seq_len, n_kv_heads, head_dim)
        n_rep: 每组的 Q 头数

    Returns:
        (batch, seq_len, n_kv_heads * n_rep, head_dim)
    """
    if n_rep == 1:
        return x
    B, T, n_kv_heads, head_dim = x.shape
    return x[:, :, :, None, :].expand(B, T, n_kv_heads, n_rep, head_dim).reshape(B, T, n_kv_heads * n_rep, head_dim)

class Attention(nn.Module):
    def __init__(self, config:LLaMaConfig):
        super().__init__()
        self.n_heads = config.n_heads
        self.n_kv_heads = config.n_kv_heads
        self.n_rep = self.n_heads // self.n_kv_heads
        self.head_dim = config.head_dim
        self.hiiden_dim = config.hidden_dim

        self.q = nn.Linear(config.hidden_dim, config.n_heads * self.head_dim, bias=False)
        self.k = nn.Linear(config.hidden_dim, config.n_kv_heads * self.head_dim, bias=False)
        self.v = nn.Linear(config.hidden_dim, config.n_kv_heads * self.head_dim, bias=False)
        self.output = nn.Linear(config.n_heads * self.head_dim, config.hidden_dim, bias=False)

    def forward(self, x:torch.Tensor, freqs_cis:torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        B, T, _ = x.shape

        q = self.q(x).view(B, T, self.n_heads, self.head_dim)
        k = self.k(x).view(B, T, self.n_kv_heads, self.head_dim)
        v = self.v(x).view(B, T, self.n_kv_heads, self.head_dim)

        #rotate
        q = apply_rotary_emb(q, freqs_cis)
        k = apply_rotary_emb(k, freqs_cis)

        #expand
        k = repeat_kv(k, self.n_rep)
        v = repeat_kv(v, self.n_rep)
        #(B, seq-len, n_heads, head_dim)

        #transpose
        q = q.transpose(1, 2).contiguous()
        k = k.transpose(1, 2).contiguous()
        v = v.transpose(1, 2).contiguous()
        #(B, n_heads, seq-len, head_dim)

        scores = (q @ k.transpose(-1, -2)) / math.sqrt(self.head_dim)
        if mask is not None:
            scores = scores + mask

        attention_weigth = F.softmax(scores.float(), dim=-1).type_as(q)
        #(B, n_heads, seq-len, seq-len)
        output = attention_weigth @ v
        #(B, n_heads, seq-len, head_dim)

        ouput = output.transpose(1, 2).contiguous().reshape(B, T, -1)
        return self.output(ouput)

#### TransformersBlock

In [7]:
class TransformersBlock(nn.Module):
    def __init__(self, config:LLaMaConfig):
        super().__init__()
        self.pre_attention_norm = RMSNorm(config.hidden_dim, config.norm_eps)
        self.attention = Attention(config)
        self.pre_FFN_norm = RMSNorm(config.hidden_dim, config.norm_eps)
        self.FFN = SwiGlu_FFN(config.hidden_dim, config.ffn_dim)

    def forward(self, x:torch.Tensor, freqs_cis:torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        attention_input = self.pre_attention_norm(x)
        attention_output = self.attention(attention_input, freqs_cis, mask)

        ffn_input = self.pre_FFN_norm(x + attention_output)
        ffn_output = self.FFN(ffn_input)
        TransformersBlock_output = (ffn_output + attention_output)
        return TransformersBlock_output

#### LLaMa Architecture

In [8]:
class LLaMa(nn.Module):
    def __init__(self, config:LLaMaConfig):
        super().__init__()
        self.config = config

        self.embedding = nn.Embedding(config.vocab_size, config.hidden_dim)
        self.dense_transfomers = nn.ModuleList([TransformersBlock(config) for _ in range(config.n_layers)])
        self.norm = RMSNorm(config.hidden_dim, config.norm_eps)
        self.output = nn.Linear(config.hidden_dim, config.vocab_size)

        self.freqs_cis = computer_freqs_cis(config.head_dim, config.max_seq_length)

        self.apply(self._init_weight)

    def _init_weight(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_idx:torch.Tensor, target:Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        Args:
            input_idx: (batch, seq_len) token IDs
            targets: (batch, seq_len) target IDs (for training)

        Returns:
            logits/output token: (batch, seq_len, vocab_size)
            loss: scalar (if targets provided)
        """
        B, T = input_idx.shape
        assert T <= self.config.max_seq_len

        # Token Embedding（无位置编码！）
        token_ids = self.embedding(input_idx)

        freqs_cis = self.freqs_cis[:T].to(token_ids.device)

        mask = torch.full((4, 4), float('-inf'), device=device)
        attention_mask = torch.triu(mask, diagonal=1)

        for layer in self.dense_transfomers:
            x = layer(token_ids, freqs_cis, attention_mask)

        output = self.norm(x)
        logits = self.output(output)
        # (B, T, vocab_size)
        loss = None
        if target is not None:
        ## Because F.cross_entropy expects:
        ## Input: (N, C) where N is the number of samples, C is the number of classes
        ## Target: (N,) the true class index for each sample
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                target.view(-1),
                ignore_index=-1,
            )
        return logits, loss

    @torch.no_grad()
    def generate(self, input_idx:torch.Tensor, max_new_tokens: int,   temperature: float = 1.0, top_k: Optional[int] = None,) -> torch.Tensor:
        for _ in range(max_new_tokens):
            idx = input_idx if input_idx.size(1) <= self.config.max_seq_len else input_idx[:, -self.config.max_seq_len:]
            #input_idx: (batch, seq_len)
            logits, _ =self.forward(idx)
            logits = logits[:, -1, :] / temperature
            # shape: (batch_size, vocab_size)

            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')

            probs = F.softmax(logits, dim=-1)
            # Sample randomly according to the probability distribution
            idx_next = torch.multinomial(probs, num_samples=1)
            # idx_next.shape = (batch_size, 1)
            idx = torch.cat([idx, idx_next], dim=1)
            # idx.shape = (batch_size, seq_len + 1)
            return idx

In [9]:
config = LLaMaConfig(
    vocab_size=32000,
    hidden_dim=4096,
    n_layers=32,
    n_heads=32,
    head_dim=128,
    max_seq_length=2048,
    norm_eps=1e-6,
    n_kv_heads= 32
)
model = LLaMa(config)

# 方法1: 简单统计
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Total parameters: 6,738,447,616
Trainable parameters: 6,738,447,616
